# https://news.naver.com/
## 위 사이트에서 뉴스기사 10개를 선택하여 전처리를 수행하세요. (문서 별 토큰 저장)

### 1. TDM 생성

각 문서별로 단어의 빈도를 계산하여 TDM을 생성하세요

문서 수가 고정되어 있을 때, TDM의 크기를 줄이기 위한 전처리 방법은 어떤 것들이 있나요?


### 2. TF-IDF 계산

앞에서 만든 TDM을 기반으로, 각 단어의 TF-IDF 점수를 계산해보세요.

TF-IDF가 TDM보다 유리한 점은 무엇인가요? TF와 IDF 각각의 의미에 대해 고민하여 작성해보세요.


### 3. N-gram

뉴스기사를 전처리하여 얻은 결과물을 통해 2-gram, 3-gram을 생성하고, 세 개의 ngram을 활용하여 TDM을 생성해보세요.

N-gram의 크기를 선택할 때 고려해야 할 사항은 무엇인가요? n이 커지면 어떤 장단점과 연계하여 고민해보세요.


### 4. 단어-문맥 동시등장 행렬 생성

전처리한 뉴스기사 중 하나를 선택하여 정해진 문맥 크기만큼 이동하면서 동시 등장한 단어를 카운트하는 행렬을 생성해보세요.

window_size는 단어-문맥 동시등장 행렬에서 중요한 매개변수입니다. window_size를 설정할 때 어떤 점을 고려해야 할까요? window_size가 크거나 작을 때 각각 어떤 영향을 미칠 수 있을까요?

=========================================================================================
1. 뉴스기사 10개 전처리 실행

In [1]:
import requests
from bs4 import BeautifulSoup
from konlpy.tag import Okt

In [2]:
def preprocess_naver_news(url):
    try:
        # 뉴스 HTML 가져오기
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36'}
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 본문 추출
        news_article = soup.select_one('#newsct_article')
        if not news_article:
            return None
        
        text = news_article.get_text(strip=True)
        
        # Okt 형태소 분석기 선언 및 품사 태깅 (원형 복원 변수 stem=True 설정)
        okt = Okt()
        tokens_with_pos = okt.pos(text, stem=True)
        
        # 불용어 품사 및 불용어 단어 정의
        stop_pos = ['Suffix', 'Punctuation', 'Josa', 'Foreign', 'Alpha', 'Number']
        stop_words = [
            '.', ',', '…', '·', '■', '↓', '“', '”', '~', '(', ')',
            '명', '은', '내', '를', '이', '국', '인', '으로', '보다', '가',
            '했습니다', '등', '도', '석', '에', '을', '편', '와', '까지', 
            '로', '계', '제', '한', '된', '는', '에는', '에서는', '의', '고', '지',
            '하는', '하고', '있습니다', '같은', '찾은', '됐습니다', '줄었습니다', '기록', '했구'
        ]
        
        # 불용어 처리 및 2글자 이상 단어 필터링
        final_tokens = []
        for word, pos in tokens_with_pos:
            # 오직 명사(Noun), 동사(Verb), 형용사(Adjective)만 통과시킵니다. (조사, 구두점 자동 탈락)
            if pos in ['Noun', 'Verb', 'Adjective']:
            # 불용어가 아니고 '두 글자 이상'인 단어만 저장합니다.
                if word not in stop_words and len(word) > 1:
                    final_tokens.append(word)
                    
        return final_tokens
    
    except Exception as e:
        print(f"URL 전처리 중 오류 발생 ({url}): {e}")
        return None

In [3]:
news_urls = [
    'https://n.news.naver.com/mnews/article/020/0003751354',
    'https://n.news.naver.com/mnews/article/003/0014220812',
    'https://n.news.naver.com/mnews/article/119/0003137435',
    'https://n.news.naver.com/mnews/article/032/0003473077',
    'https://n.news.naver.com/mnews/article/088/0001031147',
    'https://n.news.naver.com/mnews/article/001/0016343513',
    'https://n.news.naver.com/mnews/article/382/0001296293',
    'https://n.news.naver.com/mnews/article/001/0016343414',
    'https://n.news.naver.com/mnews/article/119/0003137426',
    'https://n.news.naver.com/mnews/article/003/0014220428'
]

In [5]:
documents_tokens = {}

for i, url in enumerate(news_urls):
    print(f"[{i+1}/10] 전처리 중인 뉴스: {url}")
    
    # 함수 호출하여 해당 문서의 최종 토큰 리스트 받기
    tokens = preprocess_naver_news(url)
    
    if tokens:
        # 'doc_1', 'doc_2'형식의 Key 값으로 문서별 토큰 리스트 저장
        documents_tokens[f'doc_{i+1}'] = tokens
    else:
        print(f" -> [실패] {i+1}번째 뉴스")

print("--- 전처리 및 토큰 저장 완료 ---")

# 결과 확인 (일부 출력 테스트)
if 'doc_1' in documents_tokens:
    print(documents_tokens['doc_1'][:20])

[1/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/020/0003751354
[2/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/003/0014220812
[3/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/119/0003137435
[4/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/032/0003473077
[5/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/088/0001031147
[6/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/001/0016343513
[7/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/382/0001296293
[8/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/001/0016343414
[9/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/119/0003137426
[10/10] 전처리 중인 뉴스: https://n.news.naver.com/mnews/article/003/0014220428
--- 전처리 및 토큰 저장 완료 ---
['최종', '설치미술가', '작품', '우주', '너머', '작가', '제공', '설치미술가', '최종', '서울', '노들섬', '열리다', '서울', '라이트', '한강', '빛섬', '축제', '우주', '너머', '선보이다']


======================================================================================

2. TDM 생성

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [7]:
corpus = []
doc_names = []

for doc_id, tokens in documents_tokens.items():
    # 토큰들을 공백(' ')으로 이어 붙여 하나의 문장처럼 만듦
    corpus.append(" ".join(tokens))
    doc_names.append(doc_id)  # ['doc_1', 'doc_2', ..., 'doc_10']

In [8]:
# sklearn의 CountVectorizer를 활용하여 DTM(Document-Term Matrix) 생성

count_vect = CountVectorizer(token_pattern=r'(?u)\b\w+\b') 
DTM = count_vect.fit_transform(corpus)    # 단어 사전을 만들고(fit), 빈도 수 채워넣기(transform)

# DTM을 전치(Transpose)하여 TDM(Term-Document Matrix)으로 변환
TDM_matrix = DTM.toarray().T

# 단어(Vocab) 목록을 인덱스로 하는 판다스 데이터프레임 생성
vocab = count_vect.get_feature_names_out()    # 단어 사전 뽑아오기
df_TDM = pd.DataFrame(TDM_matrix, columns=doc_names, index=vocab)

# 생성된 TDM 결과 확인
print("--- 10개 문서 기반의 TDM 생성 완료 ---")
display(df_TDM.head(20))  # 상위 20개 단어 빈도 확인

--- 10개 문서 기반의 TDM 생성 완료 ---


,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,doc_10
가격,0,0,0,0,0,0,0,0,1,0
가까이,0,0,0,0,0,0,0,0,0,1
가깝다,0,0,0,1,0,0,0,0,1,0
가능하다,0,0,2,0,0,0,0,0,0,0
가다,0,0,1,0,1,1,0,0,0,1
가사,0,0,0,0,0,0,1,0,0,0
가수,0,0,0,0,0,0,0,0,1,0
가운데,0,0,1,0,0,0,0,0,0,0
가을,0,0,0,0,0,0,0,0,0,2
가장,0,0,2,0,0,0,0,0,2,0


[문서 수가 고정되어 있을 때, TDM 크기를 줄이기 위한 전처리 방법]

- TDM크기는 고유 단어 수(행의 수) * 문서 수(열의 수) 이므로, 고유 단어 수를 줄이는 게 필요하다.

- 불용어 제거 강화, 품사 필터링, 원형 복원(표준화), 단어 길이 제한(2글자 이상만 추출), 빈도수 기반 필터링(최소빈도/최대빈도 단어 제한) 등이 있음

===============================================================================

3. TF-IDF 계산

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfTransformer

In [11]:
tfidf_trans = TfidfTransformer()

# 앞 단계의 빈도 결과(DTM)를 입력하여 TF-IDF 가중치 계산 행렬 생성
tfidf = tfidf_trans.fit_transform(DTM)

# TDM 형태로 변환하기 위해 전치(.T) 연산 수행
tfidf_matrix = tfidf.toarray().T

# 판다스 데이터프레임으로 변경
df_tfidf = pd.DataFrame(tfidf_matrix, columns=doc_names, index=vocab).round(6)

# 최종 TF-IDF 계산 결과 확인
print("--- 10개 문서 기반의 TF-IDF 행렬 생성 완료 ---")
display(df_tfidf.head(20)) # 상위 20개 단어의 TF-IDF 점수 확인

--- 10개 문서 기반의 TF-IDF 행렬 생성 완료 ---


,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,doc_10
가격,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.026740,0.000000
가까이,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.039294
가깝다,0.0,0.0,0.000000,0.036498,0.000000,0.000000,0.000000,0.000000,0.022731,0.000000
가능하다,0.0,0.0,0.036973,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
가다,0.0,0.0,0.012224,0.000000,0.026790,0.019923,0.000000,0.000000,0.000000,0.025983
가사,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.046323,0.000000,0.000000,0.000000
가수,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.026740,0.000000
가운데,0.0,0.0,0.018486,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
가을,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078589
가장,0.0,0.0,0.031430,0.000000,0.000000,0.000000,0.000000,0.000000,0.045463,0.000000


[TF-IDF가 TDM보다 유리한 점]

- TF : 단어 빈도로써, 특정 단어가 하나의 문서 내부에서 얼마나 자주 등장하는지를 나타내는 값
- IDF : 특정 단어가 전체 문서들 사이에서 얼마나 희귀하게 등장하는지를 나타내는 값 (문서 빈도의 역수)

- 주제어 추출의 정확성 : TDM(단순 빈도)은 문서에 단어가 많이 나오면 무조건 높은 점수를 줍니다. 이 경우 뉴스 문맥상 빈번하게 등장하지만 의미 없는 단어들(예: '따르면', '대해', '가운데')이 상위권을 독점하게 됩니다. 반면 TF-IDF는 흔한 단어에 점수를 낮게하고, 기사의 핵심 주제를 담은 독창적인 단어들의 점수를 끌어올려 주므로 뉴스의 진짜 핵심 키워드를 훨씬 더 정확하게 가려낼 수 있습니다.

- 문서 간 변별력 확보 : 10개의 뉴스가 있을 때, 단순 TDM은 모든 뉴스에 흔한 단어들 때문에 문서의 특징이 뭉개지기 쉽습니다. 하지만 TF-IDF는 각 문서만이 가지는 '개성 있는 단어'에 집중하게 만들어주므로, 향후 문서 분류나 유사도 분석을 할 때 성능이 압도적으로 향상됩니다.


====================================================================================

4. N-gram 계산

In [12]:
# 2-gram 생성

vect_2gram = CountVectorizer(ngram_range=(2, 2), token_pattern=r'(?u)\b\w+\b')
vect_2gram.fit(corpus)

print(list(vect_2gram.get_feature_names_out())[:10]) # 가독성을 위해 앞의 10개만 샘플 출력
print("-" * 80)

['가격 오르다', '가까이 넓히다', '가깝다 무대', '가깝다 주장', '가능하다 미래', '가능하다 해양', '가다 글로벌', '가다 대신', '가다 이어지다', '가다 택견']
--------------------------------------------------------------------------------


In [13]:
# 3-gram 생성

vect_3gram = CountVectorizer(ngram_range=(3, 3), token_pattern=r'(?u)\b\w+\b')
vect_3gram.fit(corpus)

print(list(vect_3gram.get_feature_names_out())[:10]) # 가독성을 위해 앞의 10개만 샘플 출력
print("=" * 80)

['가격 오르다 제작사', '가까이 넓히다 하다', '가깝다 무대 오르다', '가깝다 주장 예전', '가능하다 미래 하다', '가능하다 해양 가치', '가다 글로벌 라인업', '가다 대신 바다', '가다 이어지다 멘델스존', '가다 택견 무대']


In [14]:
# ngram_range=(1, 3) 설정을 주면 1글자 단위, 2글자 묶음, 3글자 묶음 단어가 모두 하나의 사전에 병합
count_vect_combined = CountVectorizer(ngram_range=(1, 3), token_pattern=r'(?u)\b\w+\b')

# 10개 문서에 대해 세 종류의 n-gram 빈도를 동시에 계산하여 DTM 구축
DTM_combined = count_vect_combined.fit_transform(corpus)

# TDM으로 90도 회전 (.T)
TDM_combined_matrix = DTM_combined.toarray().T

# 세 종류의 n-gram이 모두 융합된 고유 단어 사전 추출
vocab_combined = count_vect_combined.get_feature_names_out()

# 최종 TDM을 판다스 데이터프레임으로 포장
df_TDM_combined = pd.DataFrame(TDM_combined_matrix, columns=doc_names, index=vocab_combined)

# 정돈된 최종 마스터 TDM 출력 확인
print(f"(총 고유 단어 및 조합 개수: {len(vocab_combined)}개)")
display(df_TDM_combined.head(25)) # 다양한 형태의 n-gram 빈도가 융합된 상위 25개 행 확인

(총 고유 단어 및 조합 개수: 7579개)


,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,doc_10
가격,0,0,0,0,0,0,0,0,1,0
가격 오르다,0,0,0,0,0,0,0,0,1,0
가격 오르다 제작사,0,0,0,0,0,0,0,0,1,0
가까이,0,0,0,0,0,0,0,0,0,1
가까이 넓히다,0,0,0,0,0,0,0,0,0,1
가까이 넓히다 하다,0,0,0,0,0,0,0,0,0,1
가깝다,0,0,0,1,0,0,0,0,1,0
가깝다 무대,0,0,0,0,0,0,0,0,1,0
가깝다 무대 오르다,0,0,0,0,0,0,0,0,1,0
가깝다 주장,0,0,0,1,0,0,0,0,0,0


[N 크기 선택 시 고려할 사항]

- 문맥의 보존력과 차원의 저주 사이의 균형을 고려해야함
- n이 커지면, 문맥과 의미를 보존할 확률이 올라감
- 하지만 n이 커질수록 vocab의 크기가 기하급수적으로 늘어나 TDM의 크기가 너무 커져 차원의 저주 문제가 발생할 수 있음
- 또, 단어가 길게 묶일수록 완벽하게 일치하는 문장을 찾을 확률이 낮아지므로, 결과적으로 행렬의 숫자가 0이 많아지는 **희소행렬**이 되어 문서 간의 공통점을 찾기 어려워지는 등 희소성 문제가 나타날 수 있음

- 따라서, n은 데이터셋의 전체 크기와 문서 간 유사도 비교 등에 필요한 정도를 고려하여 설정하는 것이 필요함 

===================================================================================

5. 단어-문맥 동시등장 행렬 생성

* 단어-문맥 동시등장 행렬 : 고유 단어 수 * 고유 단어 수 크기의 정사각형 행렬 

In [15]:
import numpy as np

In [16]:
tokens = documents_tokens['doc_1']

unique_words = sorted(list(set(tokens)))
word_to_idx = {word: i for i, word in enumerate(unique_words)}
vocab_size = len(unique_words)

# 고유 단어 수 x 고유 단어 수 크기의 빈 행렬(0으로 채워진 행렬) 생성
co_matrix = np.zeros((vocab_size, vocab_size), dtype=int)

# 설정할 문맥 크기 (window_size = 2: 앞뒤 2단어씩 조사)
window_size = 2

In [17]:
for i, current_word in enumerate(tokens):
    current_idx = word_to_idx[current_word]
    
    # 현재 단어의 왼쪽(앞)과 오른쪽(뒤) 범위 지정
    start = max(0, i - window_size)
    end = min(len(tokens), i + window_size + 1)
    
    # 윈도우 범위 내의 주변 단어들을 조사
    for j in range(start, end):
        if i == j: 
            continue # 자기 자신은 제외
            
        context_word = tokens[j]
        context_idx = word_to_idx[context_word]
        
        # 동시 등장 빈도 1 증가
        co_matrix[current_idx, context_idx] += 1

# 판다스 데이터프레임으로 변환
df_co_matrix = pd.DataFrame(co_matrix, index=unique_words, columns=unique_words)

# 최종 동시등장 행렬 결과 확인
print(f"행렬 크기: {df_co_matrix.shape}")
display(df_co_matrix.head(20)) # 가독성을 위해 상위 20개 단어 간의 동시등장 빈도 확인

행렬 크기: (113, 113)


,개별,경험,고요하다,고요한,공간,공연,관객,관람객,관심,구현,...,팽팽하,풍경,하다,한강,행사,형상,형태,확대,확장,활용
개별,0,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
경험,0,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
고요하다,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
고요한,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
공간,1,1,0,0,0,0,0,0,1,0,...,0,0,1,0,0,1,0,1,2,0
공연,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
관객,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
관람객,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
관심,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
구현,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


[window_size 설정 시 주요 고려사항]

- 텍스트 특성과 분석 목적
    - 분석하고자 하는 텍스트가 짧은 문장 중심(예: 트위터, 댓글)인지, 호흡이 긴 신문 기사나 논문인지에 따라 결정해야 합니다.
    - 또한, 단순히 문법적/통사적 유사성을 보고 싶은지, 아니면 넓은 의미의 주제적/내용적 연관성을 포착하고 싶은지에 따라 윈도우 크기를 조율해야 합니다.

- size가 작으면
    - 바로 옆이나 근처에 붙어 있는 단어들만 강하게 카운트하므로, 직접적인 수식 관계나 문법적 유사성(통사적 정보)을 파악하는 데 매우 유리합니다.
    - 하지만, 문장 내에서 거리가 3~4단어 이상 조금만 떨어져 있어도 같은 도메인이나 주제를 공유하는 단어들의 연관성을 전혀 포착하지 못하는 한계가 있습니다.

- size가 크면
    - 주제론적(Topical) 유사성을 포착하기 좋습니다.
    - 하지만, 아무런 의미적 연관성이 없는 무작위 단어들까지도 단순히 '같은 문장에 존재한다'는 이유로 동시등장 빈도가 올라가 행렬에 심한 **노이즈(Noise)**가 낍니다. 이로 인해 단어 고유의 세밀한 문법적 특징이 뭉개지고, 행렬의 수치들이 전반적으로 상향 평준화되어 변별력이 떨어질 수 있습니다.
